In [1]:
import os
import scanpy as sc
from annealing_functions import annealing_process, quantum_test
from mi_functions import mutual_information_matrix
from process_data import mat_transform_sc # For single cell adata object
#from process_data import mat_transform # for an n x m matrix
from qfeatures import annealing_process, calculate_mi, preprocess_data


In [2]:
# Download h5ad here: https://drive.google.com/drive/folders/1-2og2FAM0_6e3L2_9C7HnOm9sgfrJIRe?usp=sharing
# Load data
base_path = r"C:\Users\ssromerogon\Documents\vscode_working_dir\QUBO_Feature_Selection\qubo_fs_dwave\efficient_differentiation\5000_HVGs"
fname = r"Data_hESC_EC_day1_5000g_filtered_feature_bc_matrix_h5.h5ad"

# Use os.path.join for cross-platform compatibility
file_dir = os.path.join(base_path, fname)

# Read the AnnData object
adata = sc.read_h5ad(file_dir)

adata.layers['counts'] = adata.X.copy() 
# Print the AnnData object summary
print(adata)

AnnData object with n_obs × n_vars = 4697 × 5000
    obs: 'CellID', 'BatchID', 'ClusterID', 'CellType', 'CellCycle', 'cell_potency', 'monocle3_pseudotime', 'splinefit_pseudotime', 'Tmonocleout'
    layers: 'counts'


In [ ]:
Xy = mat_transform_sc(adata, obs_key='monocle3_pseudotime')
# Xy = mat_transform(X, y) # For an n x m matrix and a target prediction vector

(5000, 4697)
(1, 4697)
Final shape of Xy: (5001, 4697)


In [4]:
# n_jobs=-1 works with all available processors
MI_mat = mutual_information_matrix(Xy, bins_method='rice', n_jobs=-1)

Elapsed time for MI construction: 914.0649 seconds


In [9]:
# Look for K features
alphasol, xsol = annealing_process(MI_mat, K=100)

Solution's energy 9.000359568744898e-09 with alpha 0.0 and 1 features
Solution's energy -605.7357724436584 with alpha 1.0 and 5000 features
Solution's energy -4.726462112615991 with alpha 0.5 and 82 features
Solution's energy -14.293549381241974 with alpha 0.75 and 214 features
Solution's energy -8.232391475532495 with alpha 0.625 and 128 features
Solution's energy -6.270376354626933 with alpha 0.5625 and 103 features
Solution's energy -5.455345323567599 with alpha 0.53125 and 93 features
Solution's energy -5.85104980406868 with alpha 0.546875 and 97 features
Solution's energy -6.057486188678013 with alpha 0.5546875 and 101 features
Solution's energy -5.953716814117797 with alpha 0.55078125 and 99 features
Solution's energy -6.0054158741531865 with alpha 0.552734375 and 99 features
Solution's energy -6.031409832396093 with alpha 0.5537109375 and 100 features
Optimal alpha value from root_scalar: 0.5537109375
Solution's energy -6.031409832396093 with alpha 0.5537109375 and 100 features


In [14]:
import pandas as pd
features = adata.var_names
df_features = pd.DataFrame(features, columns=['Gene'])  # Create a DataFrame from var_names


In [34]:
# Verify and test solution on D-Wave's system
# NOTE follow README.txt Setup D-Wave API to run mode='qa'
df_results, qa_sol = quantum_test(alphasol, MI_mat, df_features,  K=100,  mode='qa')

Quantum (Hybrid) Solver Results...
Quantum Annealing time: 0.078884 seconds
D-Wave Hybrid Solver time: 14.559107 seconds
Energy:  -6.031409828677344
Occurrences:  1


In [35]:
print(df_results)

      Gene  feature_selected  feature_score
0    YWHAB                 1      -0.164072
1   DYNLT1                 1      -0.139980
2     APLN                 1      -0.132341
3      ID1                 1      -0.131316
4    CLDN7                 1      -0.125340
..     ...               ...            ...
95   DDIT3                 1      -0.022466
96    POMC                 1      -0.022380
97     MPG                 1      -0.020759
98   SPON2                 1      -0.018699
99     NMI                 1      -0.018273

[100 rows x 3 columns]
